In [1]:
from typing import List
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

"""
표준 정답지(Golden Set)
- 질문 (user_input)
- 모범정답 (reference)
- 근거문서 (retrieved_contexts)
"""

class RagEvalSample(BaseModel):
    user_input: str = Field(description="제공된 근거 문서들을 바탕으로 사용자가 질의할 법한 질문")
    reference: str = Field(description="근거 문서에 기반한 모범정답(Ground Truth)")
    retrieved_contexts: List[str] = Field(
        description="해당 질문과 정답을 도출하는 데 실제로 사용된 1개 이상의 원본 문서 내용"
    )

class RagEvalDataset(BaseModel):
    samples: List[RagEvalSample] = Field(
        description="근거 문서로부터 생성된 평가 샘플 목록"
    )

source_docs: List[Document] = [
    Document(
        page_content="원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.",
        metadata={"source": "근태규정.pdf", "chunk_id": 1}
    ),
    Document(
        page_content="복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.",
        metadata={"source": "복리후생.pdf", "chunk_id": 2}
    ),
    Document(
        page_content="수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.",
        metadata={"source": "인사핸드북.pdf", "chunk_id": 3}
    ),
]

def format_documents(docs):
    _docs = [f"[문서 {doc.metadata.get("chunk_id")}]\n{doc.page_content.strip()}" for doc in source_docs]

    return "\n\n".join(_docs)

In [2]:
model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low"
)

structured_model = model.with_structured_output(RagEvalDataset)

prompt = PromptTemplate.from_template(
    """
    당신은 RAG 평가 데이터를 만드는 전문가입니다.
    제공된 여러 개의 [근거 문서 목록]을 분석하여 사용자가 질의할 법한 '질문'과 '정답' 쌍을 {num_samples}개 생성하세요.

    [작성 지침]
    질문은 다음과 같은 유형으로 구성하세요.
    - 단순 사실 확인: 하나의 문서를 근거해서 단순하게 답변할 수 있는 질문
    - 복합 추론 질문: 여러개의 문서를 근거하고 논리적 추론이 필요한 질문
    - 문서 외 함정 질문: 제공된 문서에 등장하지 않는 근거가 필요한 질문(환각 확인 목적)
    
    [근거 문서 목록]
    {context}
    """
)

chain = prompt | structured_model

results = chain.invoke({
    "context": format_documents(source_docs),
    "num_samples": 4
})

results

RagEvalDataset(samples=[RagEvalSample(user_input='원격 근무를 하는 날에는 몇 시까지 어디에 출근 등록을 해야 하나요?', reference='원격 근무 당일에는 오전 9시까지 사내 메신저의 근태 채널에 출근 등록을 완료해야 합니다.', retrieved_contexts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.']), RagEvalSample(user_input='수습 기간인 신규 입사자가 주 2회 재택근무를 하면서 재택근무 지원금도 받을 수 있나요?', reference='받을 수 없습니다. 원격 근무 지원 규정상 재택근무는 주 2회 가능하지만, 수습 사원 근무 가이드에 따르면 신규 입사자는 입사 후 3개월의 수습 기간 중 원격 근무와 관련 지원금 신청이 제한됩니다.', retrieved_contexts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', '수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.']), RagEvalSample(user_input='재택근무 지원금은 입사 후 첫 달부터 일할 계산되어 지급되나요?', reference='제공된 문서에는 재택근무 지원금의 입사 첫 달 일할 계산 여부가 나와 있지 않습니다. 문서에는 지원금이 월 50,000원이며 매월 25일 급여일에 합산되어 지급된다고만 명시되어 있습니다.', retrieved_contexts=['복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.']), RagEvalSample(user_input='재택근무 지원금은 얼마이며 언제 지급되나요?', reference='재택근무 지원금은 월 50,000

In [ ]:
"""
RagEvalDataset(samples=[RagEvalSample(user_input='원격 근무를 하는 날에는 몇 시까지 어디에 출근 등록을 해야 하나요?', reference='원격 근무 당일에는 오전 9시까지 사내 메신저의 근태 채널에 출근 등록을 완료해야 합니다.', retrieved_contexts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.']), 
RagEvalSample(user_input='수습 기간인 신규 입사자가 주 2회 재택근무를 하면서 재택근무 지원금도 받을 수 있나요?', reference='받을 수 없습니다. 원격 근무 지원 규정상 재택근무는 주 2회 가능하지만, 수습 사원 근무 가이드에 따르면 신규 입사자는 입사 후 3개월의 수습 기간 중 원격 근무와 관련 지원금 신청이 제한됩니다.', retrieved_contexts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', '수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.']), RagEvalSample(user_input='재택근무 지원금은 입사 후 첫 달부터 일할 계산되어 지급되나요?', reference='제공된 문서에는 재택근무 지원금의 입사 첫 달 일할 계산 여부가 나와 있지 않습니다. 문서에는 지원금이 월 50,000원이며 매월 25일 급여일에 합산되어 지급된다고만 명시되어 있습니다.', retrieved_contexts=['복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.']), RagEvalSample(user_input='재택근무 지원금은 얼마이며 언제 지급되나요?', reference='재택근무 지원금은 월 50,000원이며, 매월 25일 급여일에 합산되어 지급됩니다.', retrieved_contexts=['복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.'])])
"""